# Periodicity Discrimination

Stage 05 fits the behavioral observer models used in the manuscript. The cleaned notebook focuses on the retained manuscript analyses: per-block Rate, Timing, and Hybrid observers plus the per-point Hybrid mixture used for comparison.


## Pipeline Map

The stage asks which evidence stream best matches Slack behavioral psychometric curves:

1. Load Slack behavior and the copied manuscript discrimination outputs.
2. Compare Rate, Timing, and Hybrid observer fits when threshold weights are shared within each behavioral block.
3. Overlay model psychometric curves with behavioral data.
4. Compare the shared per-block Hybrid threshold mixture with the CV-specific per-point Hybrid mixture.


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

def _bootstrap_src(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        src_dir = candidate / 'src'
        if src_dir.exists() and (candidate / 'data').exists():
            if str(src_dir) not in sys.path:
                sys.path.insert(0, str(src_dir))
            return candidate
    raise RuntimeError('Could not locate repository root.')

REPO_ROOT = _bootstrap_src(Path.cwd())

from utils import require_data_files, data_subdir, ensure_src_on_path, find_repo_root
REPO_ROOT = find_repo_root(REPO_ROOT)
SRC_DIR = ensure_src_on_path(REPO_ROOT)
require_data_files(
    REPO_ROOT,
    [
        'data/detection/generated/cv0_based_responses.csv',
        'data/discrimination/slack',
        'data/discrimination/per-block/merged_results.csv',
        'data/discrimination/per-point/merged_results.csv',
    ],
    context='05_PeriodicityDiscrimination',
)


from discrimination import (
    DiscriminationDataConfig,
    fixed_settings_table,
    load_discrimination_data,
    run_table,
    model_comparison_pivot,
    model_comparison_table,
    per_block_regime_weight_summary,
    plot_regime_mixture_comparison,
    plot_psychometric_fits,
)

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 120)

## Configure and Load Manuscript Fits

The copied data lane contains only the manuscript analyses. The table lists the retained run families and tuned hyperparameters used in the manuscript core code.


In [ ]:
config = DiscriminationDataConfig(repo_root=REPO_ROOT)
data = load_discrimination_data(config)

display(run_table(data))
display(fixed_settings_table(data))

## Step 1: Compare Observer Families

Rate observers use rate evidence only, Timing observers use timing evidence only, and Hybrid observers allow both streams. Lower RMSE means the model's predicted psychometric curve is closer to Slack behavior on held-out Monte Carlo splits. The added block-mean rows average across frequencies and match the manuscript-level comparison.


In [ ]:
rmse_pivot = model_comparison_pivot(data)
display(rmse_pivot.round(4))

## Step 2: Psychometric Curve Matching

The manuscript psychometric figure overlays Slack behavior with the model curves. Thin lines show the non-winning per-block observers, thick lines show the winning per-block observer, and square markers show the per-point Hybrid predictions.


In [ ]:
fig = plot_psychometric_fits(data, include_per_point=True)
display(fig)
plt.close(fig)

## Step 3: Threshold-Regime Weights

The Hybrid model learns how much each spike-count threshold contributes to behavioral evidence. The per-block model shares a single mixture within a block, while the per-point Hybrid model can shift threshold mass across CV and frequency.


In [ ]:
block_regime_weights = per_block_regime_weight_summary(data)
block_weight_pivot = block_regime_weights.pivot_table(
    index=['block', 'model'],
    columns='regime_label',
    values='weight_mean',
)

display(block_weight_pivot.round(3))

## Step 4: Hybrid Regime Mixture Figure

The left panels show shared per-block threshold mixtures for Rate, Timing, and Hybrid observers. The right panels show the CV-specific Hybrid mixtures used by the per-point comparison.


In [ ]:
fig = plot_regime_mixture_comparison(data)
display(fig)
plt.close(fig)

---